In [1]:
import torch
import transformers
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

PyTorch: 2.13.0+cpu
Transformers: 5.14.1
PEFT: 0.21.1
Device: CPU


In [2]:
academic_data = [
    {
        "instruction": "What is Artificial Intelligence?",
        "response": "Artificial Intelligence is the field of computer science that enables machines to perform tasks that normally require human intelligence."
    },
    {
        "instruction": "What is Machine Learning?",
        "response": "Machine Learning is a branch of Artificial Intelligence where computers learn patterns from data and make predictions or decisions."
    },
    {
        "instruction": "What is Deep Learning?",
        "response": "Deep Learning is a type of Machine Learning that uses neural networks with multiple layers to learn complex patterns."
    },
    {
        "instruction": "What is supervised learning?",
        "response": "Supervised learning is a Machine Learning method where a model learns from labeled training data."
    },
    {
        "instruction": "What is unsupervised learning?",
        "response": "Unsupervised learning is a Machine Learning method that finds patterns in data without labeled answers."
    },
    {
        "instruction": "What is a neural network?",
        "response": "A neural network is a computing model made of connected artificial neurons that can learn patterns from data."
    },
    {
        "instruction": "What is Natural Language Processing?",
        "response": "Natural Language Processing is a field of Artificial Intelligence that helps computers understand and process human language."
    },
    {
        "instruction": "What is Generative AI?",
        "response": "Generative AI is Artificial Intelligence that can create new content such as text, images, audio, or code."
    },
    {
        "instruction": "What is overfitting?",
        "response": "Overfitting happens when a Machine Learning model learns the training data too closely and performs poorly on new data."
    },
    {
        "instruction": "What is a dataset?",
        "response": "A dataset is a collection of data used to train, test, or evaluate a Machine Learning model."
    }
]

print("Number of examples:", len(academic_data))
print("\nExample:")
print(academic_data[0])

Number of examples: 10

Example:
{'instruction': 'What is Artificial Intelligence?', 'response': 'Artificial Intelligence is the field of computer science that enables machines to perform tasks that normally require human intelligence.'}


In [3]:
from datasets import Dataset

dataset = Dataset.from_list(academic_data)

print(dataset)
print("\nFirst example:")
print(dataset[0])

Dataset({
    features: ['instruction', 'response'],
    num_rows: 10
})

First example:
{'instruction': 'What is Artificial Intelligence?', 'response': 'Artificial Intelligence is the field of computer science that enables machines to perform tasks that normally require human intelligence.'}


In [4]:
def format_example(example):
    return {
        "text": f"### Question:\n{example['instruction']}\n\n### Answer:\n{example['response']}"
    }

formatted_dataset = dataset.map(format_example)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

### Question:
What is Artificial Intelligence?

### Answer:
Artificial Intelligence is the field of computer science that enables machines to perform tasks that normally require human intelligence.


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True
)

print("Model loaded successfully!")


Loading tokenizer...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

C:\AIML\envs\DL\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sachi\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 563,200 || all params: 1,100,611,584 || trainable%: 0.0512


In [7]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed!")
print("Number of examples:", len(tokenized_dataset))

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenization completed!
Number of examples: 10


In [8]:
def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)

print("Labels added successfully!")
print(tokenized_dataset[0].keys())

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Labels added successfully!
dict_keys(['instruction', 'response', 'text', 'input_ids', 'attention_mask', 'labels'])


In [9]:
def prepare_labels(example):
    labels = example["input_ids"].copy()
    
    labels = [
        token if token != tokenizer.pad_token_id else -100
        for token in labels
    ]
    
    example["labels"] = labels
    return example

tokenized_dataset = tokenized_dataset.map(prepare_labels)

print("Labels prepared for training!")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Labels prepared for training!


In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./academic_lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    use_cpu=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("Training setup completed!")

Training setup completed!


In [11]:
print("Starting LoRA training...")

train_result = trainer.train()

print("\nLoRA training completed successfully! ✅")

Starting LoRA training...


Step,Training Loss
1,1.781804
2,1.724663
3,1.947957
4,1.976241
5,2.283765
6,2.179968
7,1.525711
8,1.834932
9,1.996292
10,1.934798



LoRA training completed successfully! ✅


In [12]:
adapter_path = "./academic_lora_adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved successfully! ✅")
print("Saved at:", adapter_path)

LoRA adapter saved successfully! ✅
Saved at: ./academic_lora_adapter


In [13]:
import torch

question = "Explain the difference between Artificial Intelligence and Machine Learning."

prompt = f"""### Question:
{question}

### Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

answer = tokenizer.decode(output[0], skip_special_tokens=True)

print("Question:")
print(question)

print("\nCustomized LoRA Model Answer:")
print(answer)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Explain the difference between Artificial Intelligence and Machine Learning.

Customized LoRA Model Answer:
### Question:
Explain the difference between Artificial Intelligence and Machine Learning.

### Answer:
Artificial Intelligence (AI) is a branch of computer science that focuses on creating intelligent machines that can perform tasks that are typically done by humans. AI is based on the principles of logic, computation, and programming. Machine Learning (ML) is a subset of AI that focuses on developing algorithms that can learn from data and make predictions based on that data. ML is


In [14]:
from transformers import AutoModelForCausalLM

original_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    low_cpu_mem_usage=True
)

original_model.eval()

print("Original TinyLlama model loaded successfully! ")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Original TinyLlama model loaded successfully! 


In [15]:
question = "Explain the difference between Artificial Intelligence and Machine Learning."

prompt = f"""### Question:
{question}

### Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output = original_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

original_answer = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("Question:")
print(question)

print("\nOriginal TinyLlama Answer:")
print(original_answer)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question:
Explain the difference between Artificial Intelligence and Machine Learning.

Original TinyLlama Answer:
### Question:
Explain the difference between Artificial Intelligence and Machine Learning.

### Answer:
Artificial Intelligence (AI) is a branch of computer science that focuses on creating intelligent machines that can perform tasks that are typically done by humans. AI is based on the principles of logic, computation, and programming. Machine Learning (ML) is a subset of AI that focuses on developing algorithms that can learn from data and make predictions based on that data. ML is


In [17]:
import pandas as pd

In [18]:
comparison = {
    "Question": [
        "Explain the difference between Artificial Intelligence and Machine Learning."
    ],
    "Original Model": [
        original_answer
    ],
    "LoRA Customized Model": [
        answer
    ]
}

comparison_df = pd.DataFrame(comparison)

comparison_df

,Question,Original Model,LoRA Customized Model
0,Explain the difference between Artificial Inte...,### Question:\nExplain the difference between ...,### Question:\nExplain the difference between ...


In [19]:
comparison_df.to_csv("original_vs_lora_comparison.csv", index=False)

print("Comparison saved successfully! ✅")
print("File: original_vs_lora_comparison.csv")


Comparison saved successfully! ✅
File: original_vs_lora_comparison.csv


In [20]:
import shutil

ollama_path = shutil.which("ollama")

if ollama_path:
    print("Ollama is installed ✅")
    print("Path:", ollama_path)
else:
    print("Ollama is not installed ❌")

Ollama is installed ✅
Path: C:\Users\sachi\AppData\Local\Programs\Ollama\ollama.EXE
